# HumanAI Detect - On Isleme (Asama 2, Colab GPU)

**Amac:** `human`, `ai_raw`, `ai_humanized` siniflari icin Asama 2 on islemeyi
(Stanza dilbilimsel analiz + BERTurk pseudo-perplexity + burstiness) Colab T4 GPU
uzerinde calistirmak. Yerel CPU'da tek ornek ~200 saniye surdugu icin (maskeli-LM
perplexity yontemi) bu adim GPU gerektiriyor.

**Adimlar:**
1. GPU kontrolu
2. Google Drive baglan
3. Repo'yu klonla, bagimliliklari kur
4. Ham veriyi Drive'dan kopyala (`human.jsonl`, `ai_raw.jsonl`, `ai_humanized.jsonl`)
5. `preprocess.py --limit 500` calistir (her sinif icin 500 ornek)
6. Sonuclari Drive'a kopyala

**Onkosul:** Drive'da `MyDrive/humanai_detect_data/` klasorune su dosyalari yukle:
- `human.jsonl` (data/raw/human/human.jsonl, 500 kayit)
- `ai_raw.jsonl` (data/raw/ai_raw/ai_raw.jsonl, ilk 500'u kullanilacak)
- `ai_humanized.jsonl` (data/raw/ai_humanized/ai_humanized.jsonl, ilk 500'u kullanilacak)


In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


In [ ]:
# 2. Google Drive baglan
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3a. Repo klonla (GitHub URL'ini kendi repo'nla degistir)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect


In [ ]:
# 3b. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q

# Stanza Turkce model indir (ilk calistirmada gerekli)
import stanza
stanza.download('tr')


In [ ]:
# 3c. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')


In [ ]:
# 4. Ham veriyi Drive'dan kopyala
import shutil
from pathlib import Path

DRIVE_INPUT = '/content/drive/MyDrive/humanai_detect_data'

for label in ['human', 'ai_raw', 'ai_humanized']:
    src = Path(DRIVE_INPUT) / f'{label}.jsonl'
    dst = Path(f'data/raw/{label}/{label}.jsonl')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(src, encoding='utf-8'))
        print(f'{label}: {lines} kayit kopyalandi -> {dst}')
    else:
        print(f'!!! {label}.jsonl bulunamadi: {src} (Drive'a yuklemeyi unutma)')


In [ ]:
# 5. On isleme (her sinif icin 500 ornek) -- GPU'da calisir, dbmdz/bert-base-turkish-cased otomatik indirilir
!python scripts/preprocess.py --label all --limit 500


In [ ]:
# 6. Sonuclari Google Drive'a kopyala
import shutil
from pathlib import Path

DRIVE_OUTPUT = '/content/drive/MyDrive/humanai_detect_data/interim'
Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

for label in ['human', 'ai_raw', 'ai_humanized']:
    src = Path(f'data/interim/{label}/{label}.jsonl')
    if src.exists():
        dst = Path(DRIVE_OUTPUT) / f'{label}.jsonl'
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(src, encoding='utf-8'))
        print(f'{label}: {lines} ornek -> {dst}')
    else:
        print(f'{label}: dosya bulunamadi!')

print('
Drive kopyalama tamamlandi.')


## Drive'dan Yerel Makineye Aktarim

Colab bittikten sonra:
1. `Google Drive > humanai_detect_data/interim/` klasorunu ac
2. `human.jsonl`, `ai_raw.jsonl`, `ai_humanized.jsonl` dosyalarini indir
3. Yerel projede suraya koy (mevcut dosyalarin uzerine yaz):
   - `data/interim/human/human.jsonl`
   - `data/interim/ai_raw/ai_raw.jsonl`
   - `data/interim/ai_humanized/ai_humanized.jsonl`
4. Sonra `build_features.py` ve `train_model.py` calistirilarak 500'luk performans olcumu yapilabilir
